In [4]:
import pandas as pd
import os
from metient.util.data_extraction_util import get_adjacency_matrix_from_txt_edge_list
mcpherson_dir = '/lila/data/morrisq/divyak/projects/metient/data/mcpherson_ovarian_2016'

prevalence_df = pd.read_csv(os.path.join(mcpherson_dir, 'supplement_table_9.csv'))
prevalence_df

,patient_id,paper_id,clone_id,prevalence
0,1,ApC1,A,0.055
1,1,ApC1,B,0.936
2,1,ApC1,C,0.003
3,1,ApC1,D,0.002
4,1,ApC1,E,0.001
...,...,...,...,...
404,10,ROvA9,B,0.241
405,10,ROvA9,C,0.246
406,10,ROvA9,D,0.009
407,10,ROvA9,E,0.229


In [16]:
from metient.util.vertex_labeling_util import path_matrix
import torch
pids = [1,2,3,4,7,9,10]
chars = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N", ]
for pid in pids:
    print("Patient", pid)
    fn = os.path.join(mcpherson_dir, f'patient{pid}_tree.txt')
    A = get_adjacency_matrix_from_txt_edge_list(fn)
    #print(A)
    P = path_matrix(A, remove_self_loops=True).int()
    #print(P)
    for i in range(P.shape[0]):
        # print("node", i, P[i])
        # Get child indices for this node
        descendants = torch.where(P[i] == 1)[0]
        # Get prevalence data for this clone and its descendants
        clone_prevalences = prevalence_df[
            (prevalence_df['patient_id'] == pid) & 
            (prevalence_df['clone_id'].isin([chars[i]] + [chars[d] for d in descendants]))
        ]
        
        # Group by paper_id and calculate total prevalence for clone + children
        total_prev = clone_prevalences.groupby('paper_id')['prevalence'].sum().reset_index()
        
        # Get just this clone's prevalence by paper_id 
        clone_prev = clone_prevalences[clone_prevalences['clone_id'] == chars[i]].copy()
        clone_prev = clone_prev[clone_prev['prevalence'] > 0.01]
        
        # Merge with totals and filter where clone is >50% of total
        clone_prev = clone_prev.merge(total_prev, on='paper_id', suffixes=('', '_total'))
        clone_prev = clone_prev[clone_prev['prevalence'] >= 0.5 * clone_prev['prevalence_total']]
        
        # Get paper_ids that meet criteria
        paper_ids = clone_prev['paper_id'].tolist()
        print(f"Clone {chars[i]} is in: {paper_ids}")
        clone_label = chars[i]
    print()



Patient 1
Clone A is in: ['LFTB4', 'RFTA16']
Clone B is in: ['ApC1', 'LFTB4', 'LOvB2', 'Om1', 'SBwlE4']
Clone C is in: ['SBwl', 'SBwlE4']
Clone D is in: ['LOvB2']
Clone E is in: ['ROv4']
Clone F is in: []
Clone G is in: ['ROv1']
Clone H is in: ['LOvB2', 'ROv1', 'ROv2', 'ROv4']
Clone I is in: ['ROv3', 'ROvA4']

Patient 2
Clone A is in: []
Clone B is in: ['Om1', 'Om2']
Clone C is in: []
Clone D is in: ['ROv2']
Clone E is in: []
Clone F is in: ['ROv1', 'ROv2']

Patient 3
Clone A is in: ['LOvC5', 'RFTA2']
Clone B is in: ['CDSB1', 'ClnE1', 'Om1', 'OmF2']
Clone C is in: ['ClnE1', 'LOvC5']
Clone D is in: ['CDSB1', 'ClnE1', 'LFTC1', 'LOvC5', 'OmF2', 'RFTA2', 'ROv2', 'ROvA7']
Clone E is in: ['ROv1', 'ROvA7']
Clone F is in: ['LFTC1', 'LOvC5', 'Om1', 'OmF2']
Clone G is in: ['Adnx']

Patient 4
Clone A is in: []
Clone B is in: ['LOvB2', 'LPvS', 'LPvSC1', 'ROvA5', 'RPvSD1']
Clone C is in: []
Clone D is in: ['ROv4', 'ROvA5']
Clone E is in: ['ROv4']
Clone F is in: ['ROv2']
Clone G is in: ['ROv3']
Clon